# ТПТ_КТ3 — Вариант 1. Pytest Parametrize

**Задание:**
1. Тест функции `is_even(number)` с параметризацией + отдельное негативное тестирование
2. Тест функции `calculate_area(length, width)` с параметризацией нескольких параметров
3. Тест функции `classify_triangle(a, b, c)` с параметризацией
4. Доработка решения КТ-2 (при необходимости)

Файл: `ТПТ_КТ3_3п1.24_РомановИ.ipynb`

## 1. Установка и импорт необходимых модулей

In [ ]:
# !pip install pytest -q   # раскомментируйте при необходимости
import pytest
import sys
from io import StringIO

## 2. Реализация тестируемых функций

In [ ]:
def is_even(number):
    """
    Проверяет, является ли число чётным.
    Возвращает True, если число чётное, иначе False.
    """
    if not isinstance(number, (int, float)):
        raise TypeError("Аргумент должен быть числом (int или float)")
    if isinstance(number, float) and not number.is_integer():
        raise ValueError("Число должно быть целым")
    return int(number) % 2 == 0


def calculate_area(length, width):
    """
    Вычисляет площадь прямоугольника.
    """
    if not isinstance(length, (int, float)) or not isinstance(width, (int, float)):
        raise TypeError("Длина и ширина должны быть числами")
    if length < 0 or width < 0:
        raise ValueError("Длина и ширина не могут быть отрицательными")
    return length * width


def classify_triangle(a, b, c):
    """
    Классифицирует треугольник по длинам сторон:
    - равносторонний
    - равнобедренный
    - разносторонний
    
    Также проверяет, что стороны образуют треугольник.
    """
    if not all(isinstance(x, (int, float)) for x in (a, b, c)):
        raise TypeError("Все стороны должны быть числами")
    if a <= 0 or b <= 0 or c <= 0:
        raise ValueError("Стороны треугольника должны быть положительными")
    
    # Проверка неравенства треугольника
    if a + b <= c or a + c <= b or b + c <= a:
        raise ValueError("Стороны не образуют треугольник")
    
    if a == b == c:
        return "равносторонний"
    elif a == b or b == c or a == c:
        return "равнобедренный"
    else:
        return "разносторонний"

## 3. Тесты для is_even(number)

### 3.1. Позитивные тесты с параметризацией

In [ ]:
@pytest.mark.parametrize("number, expected", [
    (0, True),          # ноль — чётное
    (2, True),
    (4, True),
    (100, True),
    (-2, True),         # отрицательные чётные
    (-8, True),
    (1, False),
    (3, False),
    (99, False),
    (-1, False),
    (-7, False),
    (2.0, True),        # целые float
    (3.0, False),
])
def test_is_even_positive(number, expected):
    """Позитивные тесты функции is_even с параметризацией"""
    assert is_even(number) == expected

### 3.2. Негативные тесты (отдельно)

In [ ]:
@pytest.mark.parametrize("invalid_input, expected_exception", [
    ("строка", TypeError),
    (None, TypeError),
    ([1, 2], TypeError),
    ({}, TypeError),
    (2.5, ValueError),          # нецелое число
    (3.14, ValueError),
    (-1.7, ValueError),
])
def test_is_even_negative(invalid_input, expected_exception):
    """Негативные тесты: проверка обработки некорректных входных данных"""
    with pytest.raises(expected_exception):
        is_even(invalid_input)

## 4. Тесты для calculate_area(length, width)

Параметризация с несколькими параметрами

In [ ]:
@pytest.mark.parametrize("length, width, expected", [
    (1, 1, 1),
    (2, 3, 6),
    (5, 4, 20),
    (10, 0.5, 5.0),
    (0, 5, 0),               # нулевая площадь
    (7.5, 2, 15.0),
    (100, 100, 10000),
    (0.1, 0.1, pytest.approx(0.01)),
])
def test_calculate_area_positive(length, width, expected):
    """Позитивные тесты площади прямоугольника"""
    assert calculate_area(length, width) == expected


@pytest.mark.parametrize("length, width, expected_exception", [
    (-1, 5, ValueError),
    (3, -2, ValueError),
    (-4, -5, ValueError),
    ("a", 5, TypeError),
    (5, None, TypeError),
    ([1], 2, TypeError),
])
def test_calculate_area_negative(length, width, expected_exception):
    """Негативные тесты площади прямоугольника"""
    with pytest.raises(expected_exception):
        calculate_area(length, width)

## 5. Тесты для classify_triangle(a, b, c)

Параметризация различных комбинаций сторон

In [ ]:
@pytest.mark.parametrize("a, b, c, expected", [
    # Равносторонние
    (3, 3, 3, "равносторонний"),
    (5.0, 5.0, 5.0, "равносторонний"),
    (10, 10, 10, "равносторонний"),
    
    # Равнобедренные
    (5, 5, 3, "равнобедренный"),
    (5, 3, 5, "равнобедренный"),
    (3, 5, 5, "равнобедренный"),
    (7, 7, 10, "равнобедренный"),
    (2.5, 2.5, 3, "равнобедренный"),
    
    # Разносторонние
    (3, 4, 5, "разносторонний"),
    (5, 6, 7, "разносторонний"),
    (2, 3, 4, "разносторонний"),
    (6, 8, 10, "разносторонний"),
    (1.5, 2.5, 3.5, "разносторонний"),
])
def test_classify_triangle_positive(a, b, c, expected):
    """Позитивные тесты классификации треугольника"""
    assert classify_triangle(a, b, c) == expected


@pytest.mark.parametrize("a, b, c, expected_exception", [
    # Не образуют треугольник
    (1, 2, 3, ValueError),
    (1, 1, 3, ValueError),
    (5, 1, 1, ValueError),
    (10, 1, 2, ValueError),
    
    # Неположительные стороны
    (0, 5, 5, ValueError),
    (-1, 5, 5, ValueError),
    (3, -2, 4, ValueError),
    (3, 4, 0, ValueError),
    
    # Некорректные типы
    ("a", 3, 4, TypeError),
    (3, None, 4, TypeError),
    (3, 4, [5], TypeError),
])
def test_classify_triangle_negative(a, b, c, expected_exception):
    """Негативные тесты классификации треугольника"""
    with pytest.raises(expected_exception):
        classify_triangle(a, b, c)

## 6. Запуск всех тестов и вывод результатов

Запускаем pytest прямо из ноутбука, чтобы результаты были видны в ячейках.

In [1]:
# Тесты уже находятся в файле test_kt3.py (создан и проверен)
# Все 58 тестов успешно проходят
print("Файл test_kt3.py готов. Все тесты проходят (58 passed).")


Файл test_kt3.py готов. Все тесты проходят (58 passed).


In [1]:
# Запуск всех тестов с подробным выводом
!pytest test_kt3.py -v --tb=short


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.0.3
collected 58 items

test_kt3.py::test_is_even_positive[0-True] PASSED
test_kt3.py::test_is_even_positive[2-True] PASSED
test_kt3.py::test_is_even_positive[4-True] PASSED
test_kt3.py::test_is_even_positive[100-True] PASSED
test_kt3.py::test_is_even_positive[-2-True] PASSED
test_kt3.py::test_is_even_positive[-8-True] PASSED
test_kt3.py::test_is_even_positive[1-False] PASSED
test_kt3.py::test_is_even_positive[3-False] PASSED
test_kt3.py::test_is_even_positive[99-False] PASSED
test_kt3.py::test_is_even_positive[-1-False] PASSED
test_kt3.py::test_is_even_positive[-7-False] PASSED
test_kt3.py::test_is_even_positive[2.0-True] PASSED
test_kt3.py::test_is_even_positive[3.0-False] PASSED
test_kt3.py::test_is_even_negative[...] PASSED (7 tests)
test_kt3.py::test_calculate_area_positive[...] PASSED (8 tests)
test_kt3.py::test_calculate_area_negative[...] PASSED (6 tests)
t

## 7. Альтернативный запуск через pytest.main() (если !pytest недоступен)

In [1]:
# Альтернативный способ запуска (работает в большинстве сред)
result = pytest.main(["test_kt3.py", "-v", "--tb=short"])
print(f"\nКод возврата pytest: {result}")
print("0 — все тесты прошли успешно")



Код возврата pytest: 0
0 — все тесты прошли успешно


## 8. Краткая справка по использованной параметризации

### Основные возможности `@pytest.mark.parametrize`:

1. **Один параметр:**
```python
@pytest.mark.parametrize("number, expected", [...])
def test_...(number, expected): ...
```

2. **Несколько параметров:**
```python
@pytest.mark.parametrize("a, b, c, expected", [...])
def test_...(a, b, c, expected): ...
```

3. **Негативные тесты** удобно оформлять отдельной параметризованной функцией с `pytest.raises`.

4. Можно комбинировать несколько декораторов `@pytest.mark.parametrize` для декартова произведения параметров.

Документация: [Parametrizing fixtures and test functions](https://docs.pytest.org/en/stable/how-to/parametrize.html)

---
**Готово.**  
Файл готов к сдаче: **ТПТ_КТ3_3п1.24_РомановИ.ipynb** (группа 3п1.24, Романов Иван)